# State 갱신과 리듀서

노드가 같은 상태 키를 여러 번 반환하면 값은 덮어쓸까요, 누적할까요?

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** TypedDict는 실행 중 값을 검사하는 클래스가 아니라 상태 키와 값의 형태를 코드에 표시하는 형식입니다. Annotated[list[str], operator.add]에서 첫 항목은 값의 형식이고 둘째 항목은 갱신을 합칠 함수입니다. 노드가 {'log': ['expand']}를 반환하면 런타임이 기존 log와 새 목록을 operator.add로 결합합니다.

**다른 상황에 적용.** 고객 상담 상태에서는 answer는 최신 답으로 덮어쓰고 evidence는 검색 단계마다 누적할 수 있습니다. 이때 evidence 노드는 전체 증거 목록을 다시 반환하지 않고 새로 찾은 증거만 반환해야 합니다. 그래야 같은 항목이 두 번 더해지지 않습니다.

**비교와 경계.** 리듀서는 여러 값을 무조건 보존하는 기능이 아닙니다. operator.add는 목록 순서를 이어 붙이지만 중복을 제거하거나 최신값을 고르지 않습니다. 집합 병합, 점수 최댓값, ID 기준 교체가 필요하면 그 규칙을 수행하는 별도 리듀서를 정의해야 합니다.


## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 서로 다른 갱신 규칙 선언

서로 다른 갱신 규칙 선언 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `UpdateState` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `normalize(state: UpdateState)`, `expand(state: UpdateState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `Annotated[value_type, reducer]`는 값의 형식과 새 갱신을 합칠 함수를 함께 표시합니다.


In [2]:
# 서로 다른 갱신 규칙 선언 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class UpdateState(TypedDict):
    query: str
    log: Annotated[list[str], operator.add]

def normalize(state: UpdateState) -> dict:
    return {'query': state['query'].strip(), 'log': ['normalize']}

def expand(state: UpdateState) -> dict:
    return {'query': state['query'] + ' 정책', 'log': ['expand']}

## 순차 그래프 컴파일

노드와 엣지를 연결한 뒤 실행 가능한 그래프로 컴파일합니다. 다음 셀에서 이 그래프를 호출합니다.

**이 셀의 코드 읽기.** `StateGraph(StateType)`은 상태 형식을 입력으로 받아 그래프 설계 객체를 만듭니다. `add_node(name, function)`은 그래프 이름과 실제 실행할 함수를 연결하며, 이 호출만으로 함수 본문이 실행되지는 않습니다. `add_edge(from, to)`는 두 노드의 고정 이동 순서를 등록합니다. `compile()`은 선언한 노드와 엣지로 `invoke()`할 실행 객체를 반환합니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 순차 그래프 컴파일 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder = StateGraph(UpdateState)
builder.add_node('normalize', normalize)
builder.add_node('expand', expand)
builder.add_edge(START, 'normalize')
builder.add_edge('normalize', 'expand')
builder.add_edge('expand', END)
graph = builder.compile()
print('갱신 그래프 준비 완료')

갱신 그래프 준비 완료


## 덮어쓰기와 누적 비교

덮어쓰기와 누적 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
result = graph.invoke({'query': '  도서 대출 기간  ', 'log': []})
print('최종 query:', result['query'])
print('누적 log:', result['log'])
assert result['query'] == '도서 대출 기간 정책'
assert result['log'] == ['normalize', 'expand']

최종 query: 도서 대출 기간 정책
누적 log: ['normalize', 'expand']


## 초기 기록을 포함해 다시 실행

초기 기록을 포함해 다시 실행 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [5]:
# 초기 기록을 포함해 다시 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
second = graph.invoke({'query': '운영 시간', 'log': ['input']})
print(second)
assert second['log'] == ['input', 'normalize', 'expand']

{'query': '운영 시간 정책', 'log': ['input', 'normalize', 'expand']}
